In [1]:
setwd("../data/processed")
long_form <- read.csv("long_form.csv")

In [2]:
# Guarded so re-running the notebook doesn't reinstall/recompile ggplot2 every time
if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2")

In [3]:
library(ggplot2)

# Shared look for every plot below: centered/bold titles, room to breathe (wider margins),
# muted gridlines, bold axis titles. Reused via `+ theme_report` instead of repeating
# theme() blocks per plot.
theme_report <- theme_minimal(base_size = 14, base_family = "Helvetica") +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5, margin = margin(b = 6)),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "grey35", margin = margin(b = 14)),
    plot.margin = margin(t = 20, r = 24, b = 16, l = 20),
    axis.title.x = element_text(margin = margin(t = 10), face = "bold", size = 12),
    axis.title.y = element_text(margin = margin(r = 10), face = "bold", size = 12),
    axis.text = element_text(color = "grey25"),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "grey92"),
    legend.title = element_text(face = "bold")
  )

In [ ]:
# Education spend vs. GINI
# Reuse the model already fit in statistical-analysis.ipynb so the error band matches the
# reported regression exactly, instead of geom_smooth() silently re-fitting its own copy;
# fall back to fitting locally if that notebook hasn't been run yet
model_education <- if (file.exists("models/model_education.rds")) {
  readRDS("models/model_education.rds")
} else {
  lm(GINI ~ Education_Spend, data = long_form)
}

# 100 evenly spaced x-values spanning the observed range, so the fitted line/ribbon below is
# smooth instead of jagged between the actual (sparser) data points
grid_education <- data.frame(Education_Spend = seq(
  min(long_form$Education_Spend, na.rm = TRUE),
  max(long_form$Education_Spend, na.rm = TRUE),
  length.out = 100
))
# predict(..., interval = "confidence") returns a fit/lwr/upr matrix for each grid point;
# cbind attaches those columns to the x-values so ggplot can plot the line and its error band
pred_education <- cbind(grid_education, predict(model_education, newdata = grid_education, interval = "confidence"))

ggplot(long_form, aes(x = Education_Spend, y = GINI)) +
  geom_point(alpha = 0.6, size = 2) +
  # inherit.aes = FALSE + their own data/aes: these two layers draw the prediction grid, not
  # the raw long_form points, so they must NOT inherit the y = GINI mapping set above
  geom_ribbon(data = pred_education, aes(x = Education_Spend, ymin = lwr, ymax = upr), inherit.aes = FALSE, alpha = 0.15, fill = "#2C6E9E") +
  geom_line(data = pred_education, aes(x = Education_Spend, y = fit), inherit.aes = FALSE, color = "#2C6E9E", linewidth = 1.1) +
  labs(
    title = "Education Spending vs. Income Inequality",
    x = "Education Spending (% of GDP)",
    y = "GINI Index"
  ) +
  theme_report

In [ ]:
# Control of Corruption vs. GINI
# IMPORTANT: this is a "Control of Corruption" score, not a corruption amount -- HIGHER means
# LESS corruption (stronger anti-corruption control), LOWER means MORE corruption. The title,
# axis label, and subtitle below all spell this out explicitly so it can't be misread as
# "higher score = more corrupt."
model_corruption <- if (file.exists("models/model_corruption.rds")) {
  readRDS("models/model_corruption.rds")
} else {
  lm(GINI ~ Corruption, data = long_form)
}

# Grid of x-values for a smooth fitted line, same idea as the Education plot above
grid_corruption <- data.frame(Corruption = seq(
  min(long_form$Corruption, na.rm = TRUE),
  max(long_form$Corruption, na.rm = TRUE),
  length.out = 100
))
# fit/lwr/upr columns from predict(), attached to the grid for plotting
pred_corruption <- cbind(grid_corruption, predict(model_corruption, newdata = grid_corruption, interval = "confidence"))

ggplot(long_form, aes(x = Corruption, y = GINI)) +
  geom_point(alpha = 0.6, size = 2) +
  # inherit.aes = FALSE: these layers plot the prediction grid, not the raw points above
  geom_ribbon(data = pred_corruption, aes(x = Corruption, ymin = lwr, ymax = upr), inherit.aes = FALSE, alpha = 0.15, fill = "#2C6E9E") +
  geom_line(data = pred_corruption, aes(x = Corruption, y = fit), inherit.aes = FALSE, color = "#2C6E9E", linewidth = 1.1) +
  labs(
    title = "Control of Corruption vs. Income Inequality",
    subtitle = "Higher score = LESS corruption (stronger anti-corruption control), not more",
    x = "Control of Corruption Score (0 = high corruption, 100 = low corruption)",
    y = "GINI Index"
  ) +
  theme_report

In [ ]:
# Business Regulation Quality vs. GINI
model_regulation <- if (file.exists("models/model_regulation.rds")) {
  readRDS("models/model_regulation.rds")
} else {
  lm(GINI ~ Regulation, data = long_form)
}

# Grid of x-values for a smooth fitted line, same idea as the Education plot above
grid_regulation <- data.frame(Regulation = seq(
  min(long_form$Regulation, na.rm = TRUE),
  max(long_form$Regulation, na.rm = TRUE),
  length.out = 100
))
# fit/lwr/upr columns from predict(), attached to the grid for plotting
pred_regulation <- cbind(grid_regulation, predict(model_regulation, newdata = grid_regulation, interval = "confidence"))

ggplot(long_form, aes(x = Regulation, y = GINI)) +
  geom_point(alpha = 0.6, size = 2) +
  # inherit.aes = FALSE: these layers plot the prediction grid, not the raw points above
  geom_ribbon(data = pred_regulation, aes(x = Regulation, ymin = lwr, ymax = upr), inherit.aes = FALSE, alpha = 0.15, fill = "#2C6E9E") +
  geom_line(data = pred_regulation, aes(x = Regulation, y = fit), inherit.aes = FALSE, color = "#2C6E9E", linewidth = 1.1) +
  labs(
    title = "Business Regulation Quality vs. Income Inequality",
    subtitle = "Higher rating = better/more effective business regulatory environment",
    x = "CPIA Business Regulation Rating (1 low - 6 high)",
    y = "GINI Index"
  ) +
  theme_report

In [ ]:
# Correlation heatmap: all 4 vars used against GINI
vars_used <- c("GINI", "Education_Spend", "Corruption", "Regulation")
# pairwise.complete.obs: for each pair of variables, use every row where BOTH are non-NA,
# rather than dropping a row entirely if ANY variable in vars_used is missing for it
corr_matrix <- cor(long_form[, vars_used], use = "pairwise.complete.obs")

# as.table() + as.data.frame() reshapes the 4x4 correlation matrix into a long, 3-column
# data frame (Var1, Var2, value) -- the format geom_tile() needs, one row per cell
corr_df <- as.data.frame(as.table(corr_matrix))
names(corr_df) <- c("Var1", "Var2", "Correlation")

ggplot(corr_df, aes(Var1, Var2, fill = Correlation)) +
  geom_tile(color = "white", linewidth = 0.6) +
  geom_text(aes(label = round(Correlation, 2)), size = 4, fontface = "bold") +
  # Diverging scale anchored at 0 (white): red for negative correlation, blue for positive,
  # so color intensity/direction map directly onto correlation sign and strength
  scale_fill_gradient2(low = "firebrick", mid = "white", high = "#2C6E9E", midpoint = 0, limits = c(-1, 1)) +
  labs(
    title = "Correlation Heatmap",
    subtitle = "Corruption = Control-of-Corruption score (higher = less corrupt)",
    x = NULL, y = NULL
  ) +
  theme_report +
  theme(panel.grid = element_blank(), axis.text.x = element_text(angle = 30, hjust = 1))